In [43]:
from dotenv import load_dotenv

In [44]:
load_dotenv()

True

In [45]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4986.48it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [46]:
embeddings.embed_query("hello ai")

[-0.03338827192783356,
 0.03453977033495903,
 0.05947453901171684,
 0.059286151081323624,
 -0.06353534013032913,
 -0.06819584965705872,
 0.08823322504758835,
 0.03444082289934158,
 -0.032785236835479736,
 -0.01581495814025402,
 0.02098170481622219,
 -0.01834026351571083,
 -0.039832208305597305,
 -0.0804707407951355,
 -0.014469239860773087,
 0.033264871686697006,
 0.0142592778429389,
 -0.03404993563890457,
 -0.1429157704114914,
 -0.0230833999812603,
 -0.02138015814125538,
 0.0026335588190704584,
 -0.047292716801166534,
 -0.010752713307738304,
 -0.0686679482460022,
 0.03112499974668026,
 0.07594593614339828,
 0.0011282894993200898,
 0.011631947942078114,
 -0.036039192229509354,
 0.04483756050467491,
 0.018390731886029243,
 0.12672805786132812,
 -0.0013597523793578148,
 0.008206655271351337,
 0.06909967958927155,
 -0.08076360821723938,
 -0.05841310694813728,
 0.0537545345723629,
 0.026227528229355812,
 -0.0068285902962088585,
 -0.05635838955640793,
 0.0032929631415754557,
 -0.072501823306

In [47]:
from sklearn.metrics.pairwise import cosine_similarity

In [48]:
documents=["what is the capital of USA?",
"Who is the president of USA",
"Who is a primeminister of india"]

In [49]:
my_query="Narendra Modi is Prime minister of India?"

In [50]:
document_embedding=embeddings.embed_documents(documents)

In [51]:
query_embedding=embeddings.embed_query(my_query)

In [52]:
cosine_similarity([query_embedding],document_embedding)

array([[0.14319945, 0.29477091, 0.62682356]])

In [53]:
from sklearn.metrics.pairwise import euclidean_distances

In [54]:
euclidean_distances([query_embedding],document_embedding)

array([[1.30904589, 1.18762714, 0.86391721]])

| Metric            | Similarity Score Range | Behavior                        |
|------------------|----------------------|---------------------------------|
| Cosine Similarity | [-1, 1]              | Focuses on angle only           |
| L2 Distance       | [0, ∞)               | Focuses on **magnitude + direction** |

In [55]:
import faiss

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


In [57]:
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

In [58]:
index=faiss.IndexFlatL2(384)

In [59]:
index

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x11a193690> >

In [60]:
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [61]:
vector_store.add_texts(["AI is powerful", "AI is future", "Dogs are cute"])

['082571a4-0a7c-414e-b85d-1803386eff6d',
 '7744e311-30e0-4db8-9f7d-f245e10a04af',
 '9a8556d7-33cc-4ada-8fca-2ff4d3a9f0bc']

In [62]:
vector_store.index_to_docstore_id

{0: '082571a4-0a7c-414e-b85d-1803386eff6d',
 1: '7744e311-30e0-4db8-9f7d-f245e10a04af',
 2: '9a8556d7-33cc-4ada-8fca-2ff4d3a9f0bc'}

In [63]:
result=vector_store.similarity_search("Tell me about AI", k=2)

In [64]:
result

[Document(id='082571a4-0a7c-414e-b85d-1803386eff6d', metadata={}, page_content='AI is powerful'),
 Document(id='7744e311-30e0-4db8-9f7d-f245e10a04af', metadata={}, page_content='AI is future')]

| Feature | Flat | IVF (Inverted File Index) | HNSW (Graph-based Index) |
|---------|------|--------------------------|--------------------------|
| Type of Search | Exact | Approximate (cluster-based) | Approximate (graph-based traversal) |
| Speed | Slow (linear scan) | Fast (search only in top clusters) | Very Fast (graph walk) |

| Dataset Size | Recommended Index                  |
|-------------|-----------------------------------|
| Up to 1L    | IndexFlatL2 or IndexFlatIP        |
| Up to 1M    | IndexIVFFlat or IndexHNSWFlat     |
| > 1M        | IndexIVFPQ or IndexHNSWFlat       |

In [65]:
from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Artificial intelligence is transforming industries by enabling smarter automation.",
    metadata={"source": "article"},
)

document_5 = Document(
    page_content="Just finished a 5-mile run in the park, feeling energized and refreshed.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Stock markets showed mixed signals today as investors reacted to global events.",
    metadata={"source": "news"},
)

document_7 = Document(
    page_content="LangChain makes it easier to build applications powered by large language models.",
    metadata={"source": "blog"},
)

document_8 = Document(
    page_content="Cooking homemade pasta is both fun and rewarding when you get the recipe right.",
    metadata={"source": "blog"},
)

document_9 = Document(
    page_content="A new study reveals the benefits of meditation for mental health and focus.",
    metadata={"source": "research"},
)

document_10 = Document(
    page_content="Exploring vector databases and embeddings for efficient semantic search.",
    metadata={"source": "article"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [66]:
index=faiss.IndexFlatL2(384)
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [67]:
vector_store.add_documents(documents=documents)

['c675dbd6-fb27-47a3-a93b-5d7ee5f2f6e5',
 'ae78db8a-1070-4ca6-a292-fd2dd22bb9c6',
 'd1da1d24-edc9-451c-b001-0643ca5601e8',
 '4f930f92-5c6d-4090-aec6-8ee8c9624658',
 '44a08e61-4c60-489c-9092-06087b3f6be3',
 '475cf7c3-1c75-4835-b0bc-5dddb281d0f8',
 'e2c0f5b6-a107-4828-81a9-15f071c132c7',
 '22681a14-cbad-45d7-a789-9aef95a82152',
 '5dad65bb-a2e6-4b57-92c2-7cbecb71b490',
 '48683ee6-b9b4-411c-b0ec-dad4bf7c3ad5']

In [68]:
result=vector_store.similarity_search("langchain provides absctractions to make working with LLMs easy")

In [69]:
vector_store.similarity_search("langchain provides absctractions to make working with LLMs easy", k=2)

[Document(id='d1da1d24-edc9-451c-b001-0643ca5601e8', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='e2c0f5b6-a107-4828-81a9-15f071c132c7', metadata={'source': 'blog'}, page_content='LangChain makes it easier to build applications powered by large language models.')]

In [70]:
vector_store.similarity_search("langchain provides absctractions to make working with LLMs easy", k=7)

[Document(id='d1da1d24-edc9-451c-b001-0643ca5601e8', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='e2c0f5b6-a107-4828-81a9-15f071c132c7', metadata={'source': 'blog'}, page_content='LangChain makes it easier to build applications powered by large language models.'),
 Document(id='22681a14-cbad-45d7-a789-9aef95a82152', metadata={'source': 'blog'}, page_content='Cooking homemade pasta is both fun and rewarding when you get the recipe right.'),
 Document(id='48683ee6-b9b4-411c-b0ec-dad4bf7c3ad5', metadata={'source': 'article'}, page_content='Exploring vector databases and embeddings for efficient semantic search.'),
 Document(id='4f930f92-5c6d-4090-aec6-8ee8c9624658', metadata={'source': 'article'}, page_content='Artificial intelligence is transforming industries by enabling smarter automation.'),
 Document(id='ae78db8a-1070-4ca6-a292-fd2dd22bb9c6', metadata={'source': 'news'}, page_content='The weather for

##### Filter on the basis of metadata

In [71]:
vector_store.similarity_search("langchain provides absctractions to make working with LLMs easy", filter={"source": {"$eq":"tweet"}})

[Document(id='d1da1d24-edc9-451c-b001-0643ca5601e8', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='c675dbd6-fb27-47a3-a93b-5d7ee5f2f6e5', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(id='44a08e61-4c60-489c-9092-06087b3f6be3', metadata={'source': 'tweet'}, page_content='Just finished a 5-mile run in the park, feeling energized and refreshed.')]

In [72]:
vector_store.similarity_search("langchain provides absctractions to make working with LLMs easy", filter={"source": "news"})

[Document(id='ae78db8a-1070-4ca6-a292-fd2dd22bb9c6', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='475cf7c3-1c75-4835-b0bc-5dddb281d0f8', metadata={'source': 'news'}, page_content='Stock markets showed mixed signals today as investors reacted to global events.')]

In [73]:
result[0].metadata

{'source': 'tweet'}

In [74]:
result[0].page_content

'Building an exciting new project with LangChain - come check it out!'

##### Now we are using the vector store as a retriever so that in future we can use it in DAG

In [75]:
retriever=vector_store.as_retriever(search_kwargs={"k":3})

In [76]:
retriever.invoke("LangChain provides abstractions to make working with LLMs easy")

[Document(id='e2c0f5b6-a107-4828-81a9-15f071c132c7', metadata={'source': 'blog'}, page_content='LangChain makes it easier to build applications powered by large language models.'),
 Document(id='d1da1d24-edc9-451c-b001-0643ca5601e8', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='22681a14-cbad-45d7-a789-9aef95a82152', metadata={'source': 'blog'}, page_content='Cooking homemade pasta is both fun and rewarding when you get the recipe right.')]

In [77]:
vector_store.save_local("todays' class faiss index") # saving on disk

In [78]:
new_vector_store=FAISS.load_local(
    "todays' class faiss index", embeddings, allow_dangerous_deserialization=True
)

### Dangerous Deserialization

#### 📦 Basic Concept
- **Serialization** = save object → file  
- **Deserialization** = file → object  

👉 Deserialization means **turning saved data back into something you can use in Python.**

---

#### 🔹 Simple Explanation
- You **saved something earlier** → that’s *serialization*  
- Now you **load it back** → that’s *deserialization*  

👉 In short:  
**Deserialization = “Loading back saved data”**

---

#### 🔹 In FAISS (Your Case)

When you run:

```python
FAISS.load_local("todays' class faiss index", embeddings, allow_dangerous_deserialization=True)

In [79]:
new_vector_store.similarity_search("langchain")

[Document(id='d1da1d24-edc9-451c-b001-0643ca5601e8', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='e2c0f5b6-a107-4828-81a9-15f071c132c7', metadata={'source': 'blog'}, page_content='LangChain makes it easier to build applications powered by large language models.'),
 Document(id='22681a14-cbad-45d7-a789-9aef95a82152', metadata={'source': 'blog'}, page_content='Cooking homemade pasta is both fun and rewarding when you get the recipe right.'),
 Document(id='5dad65bb-a2e6-4b57-92c2-7cbecb71b490', metadata={'source': 'research'}, page_content='A new study reveals the benefits of meditation for mental health and focus.')]

# Creating a RAG

```
PDF 
  ↓
Chunking 
  ↓
Embedding 
  ↓
Vector DB  

User Query 
  ↓
Vector DB (Similarity Search) 
  ↓
Relevant Data 
  ↓
LLM 
  ↓
Final Answer

Step 1: Document loading

In [80]:
from langchain_community.document_loaders import PyPDFLoader

In [81]:
FILE_PATH=r"/Users/sanjuktabaruah/Desktop/coursera/Krishnaink/agenticAI/2-Langchain Basics/2.4-VectorDatabase/DATA/ResearchDiscovery&Summarization.pdf"

In [82]:
loader=PyPDFLoader(FILE_PATH)

In [83]:
pages = loader.load()

If you want it to load faster, use asynch load

In [ ]:
# pages = []
# asynch for page in loader.alazy_load():
# pages.append(page)

SyntaxError: invalid syntax (1048022870.py, line 2)

In [85]:
len(loader.load())

29

no. of pages in the PDF = 29

Step 2: Document Chunking

In [86]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [87]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [88]:
split_docs = splitter.split_documents(pages)

In [89]:
len(split_docs)

147

step 3: We will store the chunks in the vector Database

In [90]:
index = faiss.IndexFlatIP(384)
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [91]:
vector_store.add_documents(documents=split_docs)

['3515cce4-4546-4cc3-9e7d-f216c6e4d116',
 '79cb5a76-2b6f-485d-988f-c86e60bd7563',
 'afc072bc-4c2b-46c8-8356-70b74484a2dd',
 'f43002ae-6bcb-47b9-98f4-5f3cdab2b03b',
 '96cbd139-c0cf-4c3d-bb63-da157e9ac12e',
 'fb89aa99-f79d-4900-9301-5d090cb01cd3',
 '5c85f076-5e41-4be4-a88e-06e8db587168',
 '8e8072a5-c552-43a4-8bd5-4c0e33b93aba',
 'b4f831fd-24c4-4cdc-9efd-e79c7528c000',
 'd4742992-c32e-4f93-a517-45886b232ca1',
 '220ea645-85a5-48cb-98ef-0b12a7db9260',
 '69af8d03-f4d6-49b6-be51-b801a11d3c65',
 'f852b192-f436-4163-bd4b-1f7e07da122e',
 'd40e1692-8e11-418b-81f0-05a3e7b48930',
 '5242fd69-7761-44d0-a678-353bde50d34c',
 '128d8e2e-0bdf-4bf2-b786-48d41e08b088',
 '61c80159-6296-4cdc-ab59-96572bf413b3',
 'd54615c3-603b-44db-8be6-e115f9466042',
 '6f04b9ac-865e-4c0d-b78c-39829035b8a2',
 'bbb57d95-c02e-4892-90ba-cad5d9ee92ed',
 '42d08fd3-69f0-4863-89e9-d5f48b72e857',
 'fc717a98-5f1e-4b25-930b-77d3994553b1',
 '0174018f-2a3b-4006-a089-feb0a2f72e81',
 '60412e25-c26d-4875-bc31-243f287de162',
 'ca5fe053-852c-

uuid for each splitted chunk

step 4: Create the retrieval pipeline to decide how many splitted chunks stored in the vector DB to be referred when you want to anwer an user's query

In [92]:
retriever= vector_store.as_retriever(search_kwargs={"k": 10})

In [93]:
retriever.invoke("langchain provides absctractions to make working with LLMs easy")

[Document(id='8e8072a5-c552-43a4-8bd5-4c0e33b93aba', metadata={'producer': 'Skia/PDF m80', 'creator': 'Chromium + Paged.js', 'creationdate': '2024-04-30T22:57:22.857Z', 'moddate': '2024-04-30T22:57:22.857Z', 'title': 'AI and Generative AI for Research Discovery and Summarization', 'source': '/Users/sanjuktabaruah/Desktop/coursera/Krishnaink/agenticAI/2-Langchain Basics/2.4-VectorDatabase/DATA/ResearchDiscovery&Summarization.pdf', 'total_pages': 29, 'page': 1, 'page_label': '2'}, page_content='in a much more focused way than typical web searches. LLMs are particularly well-suited to language \ntranslation (e.g., the Google Translate app), so that quantitative academic researchers who do not have English \nas a first language now have an assistive tool to help translate their non-English writing into English. The \nscope of AI technology is advancing so quickly, it is difficult to keep track of the areas of application and of'),
 Document(id='f62ee546-fecf-42ae-abad-77153bf511de', metada

Step 5: Generation using LLM model

In [107]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [137]:
from langchainhub.client import Client

client = Client()
prompt = client.pull("rlm/rag-prompt")

print(prompt)

/var/folders/65/hnv0c_w96gqfc4073czdj8_w0000gn/T/ipykernel_14372/2513825847.py:4: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = client.pull("rlm/rag-prompt")
/Users/sanjuktabaruah/Desktop/coursera/Krishnaink/agenticAI/venv/lib/python3.11/site-packages/langchainhub/client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"messages": [{"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:", "input_variables": ["question", "context"], "template_format": "f-string"}}}}], "input_variables": ["question", "context"]}}


In [95]:
import pprint

In [96]:
print(type(prompt))

<class 'str'>


In [97]:
pprint.pprint(prompt)

('{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, '
 '"type": "constructor", "kwargs": {"messages": [{"id": ["langchain", '
 '"prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": '
 '"constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", '
 '"prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": '
 '{"template": "You are an assistant for question-answering tasks. Use the '
 "following pieces of retrieved context to answer the question. If you don't "
 "know the answer, just say that you don't know. Use three sentences maximum "
 'and keep the answer concise.\\nQuestion: {question} \\nContext: {context} '
 '\\nAnswer:", "input_variables": ["question", "context"], "template_format": '
 '"f-string"}}}}], "input_variables": ["question", "context"]}}')


In [138]:
type(prompt)

str

In [129]:
import ast
prompt = ast.literal_eval(prompt)

In [130]:


data = prompt

In [99]:
data

{'id': ['langchain', 'prompts', 'chat', 'ChatPromptTemplate'],
 'lc': 1,
 'type': 'constructor',
 'kwargs': {'messages': [{'id': ['langchain',
     'prompts',
     'chat',
     'HumanMessagePromptTemplate'],
    'lc': 1,
    'type': 'constructor',
    'kwargs': {'prompt': {'id': ['langchain',
       'prompts',
       'prompt',
       'PromptTemplate'],
      'lc': 1,
      'type': 'constructor',
      'kwargs': {'template': "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:",
       'input_variables': ['question', 'context'],
       'template_format': 'f-string'}}}}],
  'input_variables': ['question', 'context']}}

In [100]:
message=data['kwargs']['messages'][0]['kwargs']['prompt']['kwargs']['template']

In [101]:
message

"You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"

"You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"

Chain: Context(retriever), prompt(hub), model(OpenAi), parser(langchain)
- Runnablepass through is used to get the question in the run time

In [102]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [103]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# prompt = data

In [111]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser



In [ ]:
# Source - https://stackoverflow.com/a/77794205
# Posted by Terri Lee
# Retrieved 2026-04-05, License - CC BY-SA 4.0

rag_chain = (
    {"context": lambda x: context, "question": RunnablePassthrough()} | prompt | model
)


NameError: name 'rag_custom_prompt' is not defined

In [116]:
context_runnable = RunnableLambda(retrieve_and_format)

In [131]:
type(prompt)

dict

In [120]:
type(model)

langchain_openai.chat_models.base.ChatOpenAI

In [122]:
type(context_runnable)

langchain_core.runnables.base.RunnableLambda

In [133]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

rag_chain = (
    {
        "context": RunnableLambda(lambda x: retriever.invoke(x["question"])),
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class 'list'>

the type cast error is ecause we were using client not hub to load the prompt as it was a limitation, thus the prompt data typr was str, but while chaining it was expecting prompt datatype to be ChatPromptTemplate

In [139]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Question: {question}
Context: {context}
Answer:
""")

In [140]:
type(prompt)

langchain_core.prompts.chat.ChatPromptTemplate

In [141]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

In [142]:
rag_chain.invoke("what is llama model?")

'The Llama model refers to a type of large language model (LLM) used in AI applications, particularly for tasks like language translation and research discovery. These models leverage advanced algorithms and training methods to generate human-like text and assist in various language-related tasks. However, they also require careful oversight to ensure the reliability of their outputs.'